# Imports

In [175]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import os
from tqdm import tqdm



# Load Metadata

In [176]:
VISION_CACHE = {}
TEXT_CACHE = {}


In [177]:
meta = pd.read_csv("fusion_indexes/fusion_index_metadata.csv")

# Remove random projection rows
meta = meta[meta["projection"] != "random"]
meta.head()

CURRENT_DATASET = "Flickr8k"
BASE_DIR = "TFE_Data"
DATASETS_DIR = os.path.join(BASE_DIR, "Datasets")
IMAGE_DIR = os.path.join(BASE_DIR, "Flickr8k", "Images", "Flicker8k_Dataset")



In [178]:
# Load raw embeddings for each model
vision_models = {
    "mobilenet_v3": np.load("TFE_Data/Unimodal_Results/Flickr8k/vision/mobilenet_v3/embeddings.npy"),
    "pvt": np.load("TFE_Data/Unimodal_Results/Flickr8k/vision/pvt/embeddings.npy")
}

text_models = {
    "bert": np.load("TFE_Data/Unimodal_Results/Flickr8k/text/bert/embeddings.npy"),
    "roberta": np.load("TFE_Data/Unimodal_Results/Flickr8k/text/roberta/embeddings.npy")
}

raw_text_models = text_models  # same for captions


# Model Embedding

# Fuse

In [179]:
def prepare_query_embedding(query_emb, W, fusion, latent_dim):
    # 1. Project into shared latent space
    q_proj = query_emb @ W

    # 2. Create zero vector for missing modality
    zero = np.zeros_like(q_proj)

    # 3. Fuse
    if fusion == "concat":
        return np.concatenate([q_proj, zero], axis=1)

    if fusion == "add":
        return q_proj[:, :latent_dim] + zero[:, :latent_dim]

    if fusion == "gated":
        return 0.5 * q_proj + 0.5 * zero

    if fusion == "mul":
        return q_proj[:, :latent_dim] * zero[:, :latent_dim]

    raise ValueError("Unknown fusion")


# Retrieve

In [180]:
def retrieve_from_fusion(query_fused, F_db, top_k=10):
    sims = cosine_similarity(query_fused, F_db).flatten()
    idx = np.argsort(sims)[::-1][:top_k]
    return idx, sims[idx]


In [181]:
def image_to_text_query(image_path, fusion_row):

    # 1. Embed query image (raw unimodal)
    Xv_q = embed_image_query(image_path, fusion_row["vision_model"])

    # 2. Load projection matrix
    Wv = np.load(fusion_row["Wv_path"])

    # 3. Prepare fused query
    q_fused = prepare_query_embedding(
        Xv_q,
        Wv,
        fusion_row["fusion"],
        fusion_row["latent_dim"]
    )

    # 4. Load fused caption database
    F_cap = np.load(fusion_row["caption_index_path"])

    # 5. Retrieve
    return retrieve_from_fusion(q_fused, F_cap, top_k=50)


In [182]:
def text_to_image_query(text, fusion_row):

    # 1. Embed query text (raw unimodal)
    Xt_q = embed_text_query(text, fusion_row["text_model"])

    # 2. Load projection matrix
    Wt = np.load(fusion_row["Wt_path"])

    # 3. Prepare fused query
    q_fused = prepare_query_embedding(
        Xt_q,
        Wt,
        fusion_row["fusion"],
        fusion_row["latent_dim"]
    )

    # 4. Load fused image database
    F_img = np.load(fusion_row["image_index_path"])

    # 5. Retrieve
    return retrieve_from_fusion(q_fused, F_img, top_k=50)


# Utility Functions

In [183]:
def recall_at_k_i2t(sim_matrix, k):
    correct = 0
    for i in range(sim_matrix.shape[0]):
        top_k = np.argsort(sim_matrix[i])[::-1][:k]
        if i in top_k:
            correct += 1
    return correct / sim_matrix.shape[0]

def mrr_i2t(sim_matrix):
    rr = []
    for i in range(sim_matrix.shape[0]):
        order = np.argsort(sim_matrix[i])[::-1]
        rank = np.where(order == i)[0][0] + 1
        rr.append(1 / rank)
    return np.mean(rr)

def recall_at_k_t2i(sim_matrix, caption_to_image_idx, k):
    correct = 0
    for cap_idx in range(sim_matrix.shape[0]):
        true_img = caption_to_image_idx[cap_idx]
        top_k = np.argsort(sim_matrix[cap_idx])[::-1][:k]
        if true_img in top_k:
            correct += 1
    return correct / sim_matrix.shape[0]

def mrr_t2i(sim_matrix, caption_to_image_idx):
    rr = []
    for cap_idx in range(sim_matrix.shape[0]):
        true_img = caption_to_image_idx[cap_idx]
        order = np.argsort(sim_matrix[cap_idx])[::-1]
        rank = np.where(order == true_img)[0][0] + 1
        rr.append(1 / rank)
    return np.mean(rr)

def median_rank(sim_matrix):
    ranks = []
    for i in range(sim_matrix.shape[0]):
        order = np.argsort(sim_matrix[i])[::-1]
        rank = np.where(order == i)[0][0] + 1
        ranks.append(rank)
    return np.median(ranks)

def mean_rank(sim_matrix):
    ranks = []
    for i in range(sim_matrix.shape[0]):
        order = np.argsort(sim_matrix[i])[::-1]
        rank = np.where(order == i)[0][0] + 1
        ranks.append(rank)
    return np.mean(ranks)

def mrr(sim_matrix):
    rr = []
    for i in range(sim_matrix.shape[0]):
        order = np.argsort(sim_matrix[i])[::-1]
        rank = np.where(order == i)[0][0] + 1
        rr.append(1 / rank)
    return np.mean(rr)


In [184]:
def topk_accuracy(retrieved_indices, true_index, k):
    return 1.0 if true_index in retrieved_indices[:k] else 0.0

def dcg_at_k(retrieved_indices, true_index, k):
    for rank, idx in enumerate(retrieved_indices[:k], start=1):
        if idx == true_index:
            return 1 / np.log2(rank + 1)
    return 0.0

def ndcg_at_k(retrieved_indices, true_index, k):
    dcg = dcg_at_k(retrieved_indices, true_index, k)
    idcg = 1.0  # ideal DCG (correct item at rank 1)
    return dcg / idcg


In [185]:
def compute_mean_rank(sim_matrix):
    ranks = []
    for i in range(sim_matrix.shape[0]):
        order = np.argsort(sim_matrix[i])[::-1]
        rank = np.where(order == i)[0][0] + 1
        ranks.append(rank)
    return np.mean(ranks)

def compute_median_rank(sim_matrix):
    ranks = []
    for i in range(sim_matrix.shape[0]):
        order = np.argsort(sim_matrix[i])[::-1]
        rank = np.where(order == i)[0][0] + 1
        ranks.append(rank)
    return np.median(ranks)

def compute_mrr(sim_matrix):
    rr = []
    for i in range(sim_matrix.shape[0]):
        order = np.argsort(sim_matrix[i])[::-1]
        rank = np.where(order == i)[0][0] + 1
        rr.append(1.0 / rank)
    return np.mean(rr)


In [186]:
def benchmark_image_batch(stress_images, fusion_row, n_runs=5):
    import time
    import torch

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    Wv = np.load(fusion_row["Wv_path"])
    latent_dim = fusion_row["latent_dim"]

    latencies = []
    start_total = time.time()

    for _ in range(n_runs):
        t0 = time.time()

        for img_path in stress_images:
            # 1. Embed query image
            Xv_q = embed_image_query(img_path, fusion_row["vision_model"])

            # 2. Prepare fused query (projection happens inside)
            _ = prepare_query_embedding(
                Xv_q,
                Wv,
                fusion_row["fusion"],
                latent_dim
            )

        torch.cuda.synchronize()
        latencies.append(time.time() - t0)

    total_time = time.time() - start_total
    avg_batch_latency = np.mean(latencies)
    per_sample_latency = avg_batch_latency / len(stress_images)
    throughput = len(stress_images) / avg_batch_latency
    peak_mem = torch.cuda.max_memory_allocated() / (1024**2)

    return {
        "batch_total_time": avg_batch_latency,
        "per_sample_latency": per_sample_latency,
        "throughput_samples_per_sec": throughput,
        "peak_gpu_memory_MB": peak_mem
    }


# I to T

# Execution

In [187]:

STRESS_TEST_IMAGES = [
    os.path.join(IMAGE_DIR, fname)
    for fname in [
        "107582366_d86f2d3347.jpg",
        "127450902_533ceeddfc.jpg",
        "112178718_87270d9b4d.jpg",
        "141755290_4b954529f3.jpg",
        "171488318_fb26af58e2.jpg",
        "240583223_e26e17ee96.jpg",
        "241346317_be3f07bd2e.jpg",
        "911795495_342bb15b97.jpg",
        "2452686995_621878f561.jpg",
        "97577988_65e2eae14a.jpg"
    ]
]

# Load Flickr8k captions
df_text = pd.read_pickle("TFE_Data/Datasets/df_Flickr8k.pkl")

captions = []
caption_image_map = []

for _, row in df_text.iterrows():
    img = row["image_name"]
    for cap in row["captions"]:
        captions.append(cap)
        caption_image_map.append(img)

images = df_text["image_name"].tolist()

image_to_idx = {img: idx for idx, img in enumerate(images)}
caption_to_image_idx = [image_to_idx[img] for img in caption_image_map]


# 3. Index of images (for evaluation)
index_to_image = images

os.makedirs("retrieval_results", exist_ok=True)



In [188]:
results = []

for _, row in tqdm(meta.iterrows(), total=len(meta)):

    print(f"Evaluating {row['vision_model']} × {row['text_model']} — {row['projection']} — {row['fusion']}")

    F_img = np.load(row["image_index_path"])
    F_cap = np.load(row["caption_index_path"])

    sim_i2t = cosine_similarity(F_img, F_cap)
    sim_t2i = cosine_similarity(F_cap, F_img)

    metrics = {
        "i2t_R1": recall_at_k_i2t(sim_i2t, 1),
        "i2t_R5": recall_at_k_i2t(sim_i2t, 5),
        "i2t_R10": recall_at_k_i2t(sim_i2t, 10),
        "i2t_MRR": mrr_i2t(sim_i2t),

        "t2i_R1": recall_at_k_t2i(sim_t2i, caption_to_image_idx, 1),
        "t2i_R5": recall_at_k_t2i(sim_t2i, caption_to_image_idx, 5),
        "t2i_R10": recall_at_k_t2i(sim_t2i, caption_to_image_idx, 10),
        "t2i_MRR": mrr_t2i(sim_t2i, caption_to_image_idx),
    }

    timing = benchmark_image_batch(
        stress_images=STRESS_TEST_IMAGES,
        fusion_row=row,
        n_runs=5
    )

    results.append({
        "vision": row["vision_model"],
        "text": row["text_model"],
        "projection": row["projection"],
        "fusion": row["fusion"],
        **metrics,
        **timing
    })

df_results = pd.DataFrame(results)
df_results


  0%|          | 0/32 [00:00<?, ?it/s]

Evaluating mobilenet_v3 × roberta — pca_shared — concat


  3%|▎         | 1/32 [00:48<25:12, 48.79s/it]

Evaluating mobilenet_v3 × roberta — pca_shared — add


  6%|▋         | 2/32 [01:37<24:17, 48.60s/it]

...

Evaluating pvt × bert — cca_shared — gated


 97%|█████████▋| 31/32 [26:35<00:53, 53.09s/it]

Evaluating pvt × bert — cca_shared — mul


100%|██████████| 32/32 [27:29<00:00, 51.54s/it]


,vision,text,projection,fusion,i2t_R1,i2t_R5,i2t_R10,i2t_MRR,t2i_R1,t2i_R5,t2i_R10,t2i_MRR,batch_total_time,per_sample_latency,throughput_samples_per_sec,peak_gpu_memory_MB
0,mobilenet_v3,roberta,pca_shared,concat,0.0,0.000124,0.000371,0.000268,1.000000,1.000000,1.000000,1.000000,1.107721,0.110772,9.027547,62.889648
1,mobilenet_v3,roberta,pca_shared,add,0.0,0.000124,0.000371,0.000275,1.000000,1.000000,1.000000,1.000000,1.080204,0.108020,9.257515,62.889648
2,mobilenet_v3,roberta,pca_shared,gated,0.0,0.000124,0.000371,0.000275,1.000000,1.000000,1.000000,1.000000,1.085800,0.108580,9.209796,62.889648
3,mobilenet_v3,roberta,pca_shared,mul,0.0,0.000124,0.000247,0.000248,0.995501,0.998443,0.998813,0.996784,1.090893,0.109089,9.166799,62.889648
4,mobilenet_v3,roberta,cca_shared,concat,0.0,0.000124,0.000247,0.000266,1.000000,1.000000,1.000000,1.000000,1.570785,0.157078,6.366244,62.889648
5,mobilenet_v3,roberta,cca_shared,add,0.0,0.000124,0.000371,0.000259,1.000000,1.000000,1.000000,1.000000,1.570783,0.157078,6.366253,62.889648
6,mobilenet_v3,roberta,cca_shared,gated,0.0,0.000124,0.000371,0.000259,1.000000,1.000000,1.000000,1.000000,1.934211,0.193421,5.170067,62.889648
7,mobilenet_v3,roberta,cca_shared,mul,0.0,0.000124,0.000247,0.000257,0.999901,0.999975,1.000000,0.999936,1.565545,0.156555,6.387552,62.889648
8,mobilenet_v3,bert,pca_shared,concat,0.0,0.000124,0.000371,0.000274,0.999852,1.000000,1.000000,0.999926,1.097936,0.109794,9.108000,62.889648
9,mobilenet_v3,bert,pca_shared,add,0.0,0.000124,0.000124,0.000263,0.999827,1.000000,1.000000,0.999905,1.100960,0.110096,9.082985,62.889648


In [189]:
df_results["global_score"] = (
    df_results["i2t_R1"] * 0.5 +
    df_results["t2i_R1"] * 0.5
)

df_results.sort_values("global_score", ascending=False)


,vision,text,projection,fusion,i2t_R1,i2t_R5,i2t_R10,i2t_MRR,t2i_R1,t2i_R5,t2i_R10,t2i_MRR,batch_total_time,per_sample_latency,throughput_samples_per_sec,peak_gpu_memory_MB,global_score
0,mobilenet_v3,roberta,pca_shared,concat,0.0,0.000124,0.000371,0.000268,1.000000,1.000000,1.000000,1.000000,1.107721,0.110772,9.027547,62.889648,0.500000
1,mobilenet_v3,roberta,pca_shared,add,0.0,0.000124,0.000371,0.000275,1.000000,1.000000,1.000000,1.000000,1.080204,0.108020,9.257515,62.889648,0.500000
2,mobilenet_v3,roberta,pca_shared,gated,0.0,0.000124,0.000371,0.000275,1.000000,1.000000,1.000000,1.000000,1.085800,0.108580,9.209796,62.889648,0.500000
4,mobilenet_v3,roberta,cca_shared,concat,0.0,0.000124,0.000247,0.000266,1.000000,1.000000,1.000000,1.000000,1.570785,0.157078,6.366244,62.889648,0.500000
6,mobilenet_v3,roberta,cca_shared,gated,0.0,0.000124,0.000371,0.000259,1.000000,1.000000,1.000000,1.000000,1.934211,0.193421,5.170067,62.889648,0.500000
5,mobilenet_v3,roberta,cca_shared,add,0.0,0.000124,0.000371,0.000259,1.000000,1.000000,1.000000,1.000000,1.570783,0.157078,6.366253,62.889648,0.500000
17,pvt,roberta,pca_shared,add,0.0,0.000124,0.000247,0.000301,1.000000,1.000000,1.000000,1.000000,1.132510,0.113251,8.829946,138.037109,0.500000
28,pvt,bert,cca_shared,concat,0.0,0.000124,0.000247,0.000282,1.000000,1.000000,1.000000,1.000000,1.601206,0.160121,6.245291,138.037109,0.500000
29,pvt,bert,cca_shared,add,0.0,0.000124,0.000247,0.000280,1.000000,1.000000,1.000000,1.000000,1.584909,0.158491,6.309511,138.037109,0.500000
30,pvt,bert,cca_shared,gated,0.0,0.000124,0.000247,0.000280,1.000000,1.000000,1.000000,1.000000,1.952271,0.195227,5.122240,138.037109,0.500000
